# Day 9: Snowflake Security, Dynamic Masking & dbt Gold Marts
This notebook simulates Gold metric calculations and role-based column masking on sample records.

In [ ]:
!pip install duckdb pandas

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect(':memory:')

# 1. Simulate Gold Mart calculation (30-day readmission)
con.execute("""
CREATE TABLE encounters (
    person_id INT,
    encounter_id INT,
    admission_date DATE,
    discharge_date DATE
)
""")

con.execute("""
INSERT INTO encounters VALUES
(1, 101, '2023-01-01', '2023-01-05'), -- Index admission
(1, 102, '2023-01-20', '2023-01-22'), -- Readmission within 30 days
(2, 201, '2023-02-01', '2023-02-10'), -- Index admission
(2, 202, '2023-04-01', '2023-04-05')  -- Readmission > 30 days
""")

query_gold_mart = """
SELECT 
    e1.person_id,
    e1.encounter_id as index_encounter,
    e1.discharge_date,
    e2.encounter_id as readmission_encounter,
    e2.admission_date as readmission_date,
    CASE 
        WHEN e2.encounter_id IS NOT NULL THEN 1 ELSE 0 
    END as is_readmitted_30_days
FROM encounters e1
LEFT JOIN encounters e2 
    ON e1.person_id = e2.person_id 
    AND e2.admission_date > e1.discharge_date 
    AND e2.admission_date <= e1.discharge_date + INTERVAL 30 DAY
"""
print("Gold Mart: 30-Day Readmission Simulation")
print(con.execute(query_gold_mart).df())

# 2. Simulate Dynamic Data Masking
data = {
    'person_id': [1, 2],
    'ssn': ['123-45-6789', '987-65-4321'],
    'dob': ['1980-05-15', '1992-11-01']
}
df_patients = pd.DataFrame(data)

def apply_masking(df, role):
    df_masked = df.copy()
    if role != 'COMPLIANCE_OFFICER':
        df_masked['ssn'] = '***-**-****'
        df_masked['dob'] = pd.to_datetime(df_masked['dob']).dt.year.astype(str) + '-01-01'
    return df_masked

print("\nData viewed by COMPLIANCE_OFFICER:")
print(apply_masking(df_patients, 'COMPLIANCE_OFFICER'))

print("\nData viewed by DATA_ANALYST (Masked):")
print(apply_masking(df_patients, 'DATA_ANALYST'))